# TTS bootcamp – run 1: Task 1 (measure baseline) + Task 2 (text frontend)
Settings: **GPU T4 x1**, **Internet On**, Secret **HF_TOKEN** attached (Add-ons → Secrets). Input: the `tts-bootcamp` dataset. Then *Save Version → Save & Run All*.

In [ ]:
!nvidia-smi
import sys, platform; print(sys.version)

In [ ]:
import os, shutil, subprocess
src = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'metrics.py' in files and 'baseline' in dirs:
        src = root; break
assert src, 'kit folder (with metrics.py) not found under /kaggle/input'
W = '/kaggle/working/tts'
shutil.rmtree(W, ignore_errors=True)
shutil.copytree(src, W)
os.chdir(W)
print('kit at', src, '->', W, sorted(os.listdir(W)))

def sh(cmd, tail=60):
    print('\n$', cmd, flush=True)
    env = dict(os.environ, PYTHONPATH=os.path.join(W, 'code', 'compat'))
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    out = (r.stdout + ('\n' + r.stderr if r.returncode else '')).strip().splitlines()
    print('\n'.join(out[-tail:]))
    print('[exit code]', r.returncode, flush=True)
    return r.returncode

In [ ]:
# compatibility shim (also shipped in code/compat/): soundfile fallback for torchaudio.load
os.makedirs('code/compat', exist_ok=True)
open('code/compat/sitecustomize.py', 'w').write('"""Compatibility shim (loaded automatically when code/compat is on PYTHONPATH).\n\ntorchaudio >= 2.9 reads audio only through the optional `torchcodec` package. If that path fails,\nfall back to soundfile so IndicF5\'s reference-clip loading still works. Audio content is unchanged.\n"""\ntry:\n    import torchaudio as _ta\n\n    _orig_load = _ta.load\n\n    def _load(path, *args, **kwargs):\n        try:\n            return _orig_load(path, *args, **kwargs)\n        except Exception:\n            import soundfile as _sf\n            import torch as _torch\n            y, sr = _sf.read(str(path), dtype="float32", always_2d=True)\n            return _torch.from_numpy(y.T.copy()), sr\n\n    _ta.load = _load\nexcept Exception:\n    pass\n')
print('shim written')

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('HF login ok')

In [ ]:
# NOTE: the kit's requirements.txt pins IndicF5 to fb652a9..., which is the Hugging Face *model* revision,
# not a commit of the GitHub code repo, so `pip install -r baseline/requirements.txt` fails (git exit 128).
# We install the same dependencies explicitly and the IndicF5 code at its latest GitHub commit (13f7c4d).
# audioop-lts: pydub needs it on Python 3.13. torchcodec: torchaudio>=2.9 needs it to read wav files.
import torch
keep = f'"torch=={torch.__version__}"'
try:
    import torchaudio; keep += f' "torchaudio=={torchaudio.__version__}"'
except ImportError:
    keep += f' "torchaudio=={torch.__version__.split('+')[0]}"'
pkgs = ('"numpy>=2.1,<2.3" "transformers==4.49.0" accelerate soundfile librosa pyyaml pandas "speechbrain>=1.0" '
        'cached_path ema_pytorch hydra-core jieba pypinyin pydub torchdiffeq vocos "x_transformers>=1.31.14" '
        'transformers_stream_generator safetensors tomli click wandb datasets audioop-lts torchcodec')
sh(f'pip install -q {keep} {pkgs}', tail=15)
sh('pip install -q --no-deps "git+https://github.com/ai4bharat/IndicF5.git@13f7c4d627cc10111aea8fe9c0039462cacacdc7"', tail=15)
sh('python -c "import f5_tts, torchaudio, numpy, transformers, torch, speechbrain; print(numpy.__version__, transformers.__version__, torch.__version__, torchaudio.__version__)"')

In [ ]:
sh('python code/warmup.py')
sh('python code/warmup.py')

## Task 1 – score the provided baseline audio

In [ ]:
os.makedirs('out', exist_ok=True)
sh('python metrics.py data/baseline_val --json out/t1_baseline.json --csv out/t1_baseline.csv', tail=50)

## Task 2 – normalize text, synthesize eval (t2_norm) and the validation control (val_control)

In [ ]:
sh('python code/normalize.py --kit . --out predictions/t2_normalized.csv')
sh('python baseline/run_baseline.py --items data/eval_items.csv --out out/t2_norm --text-csv predictions/t2_normalized.csv --text-column normalized_text --seed 0', tail=8)
sh('python baseline/run_baseline.py --items data/val_items.csv --out out/val_control --text-csv predictions/t2_normalized.csv --text-column normalized_text --seed 0', tail=8)
for d in ['t2_norm', 'val_control']:
    if os.path.isdir(f'out/{d}'):
        shutil.copytree(f'out/{d}', f'predictions/{d}', dirs_exist_ok=True)
sh('python metrics.py predictions/val_control --json out/t2_val_control.json --csv out/t2_val_control.csv', tail=50)

## Check: does a fresh baseline run reproduce data/baseline_val/?

In [ ]:
sh('python baseline/run_baseline.py --items data/val_items.csv --out out/baseline_val_repro --seed 0', tail=4)
sh('python metrics.py out/baseline_val_repro --json out/repro.json --csv out/repro.csv', tail=12)

In [ ]:
import zipfile
z = zipfile.ZipFile('/kaggle/working/run1_outputs.zip', 'w', zipfile.ZIP_DEFLATED)
for base in ['predictions', 'out']:
    for root, _, files in os.walk(base):
        for f in files:
            z.write(os.path.join(root, f))
z.close()
print('wrote run1_outputs.zip', os.path.getsize('/kaggle/working/run1_outputs.zip') // 1024, 'KB')